# Gas Turbine Ambient Derating & Power Regression Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedbasra-star/gas-turbine-ambient-effects-regression/blob/main/notebooks/quickstart_demo.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-brightgreen.svg)](https://www.python.org/downloads/)

## 1. Physical & Thermodynamic Background
Gas turbine electrical power output ($PE$) is governed by the Brayton cycle net work:
$$\dot{W}_{net} = \dot{m}_{air} \cdot (w_{turbine} - w_{compressor})$$

Where compressor mass flow rate is directly proportional to ambient air density:
$$\dot{m}_{air} = \rho_{air} \cdot A \cdot v = \frac{P_{amb}}{R_{spec} \cdot T_{amb}} \cdot A \cdot v$$

When ambient temperature $T_{amb}$ rises:
1. **Air density decreases**, reducing the mass of oxygen entering the combustion chamber per unit time.
2. **Specific compressor work** increases: $w_c = c_p \cdot T_{amb} \cdot \left( r_p^{\frac{\gamma - 1}{\gamma}} - 1 \right)$.

This leads to severe **power derating** (typically $\approx -2.1\text{ MW} / ^\circ\text{C}$ drop). In addition, condenser backpressure ($V$), atmospheric pressure ($AP$), and humidity ($RH$) modulate output.

In this notebook, we benchmark classical ML and tree-based ensembles (Ridge, Random Forest, XGBoost) to model and predict power derating.

In [ ]:
# Install dependencies if running on Google Colab
import sys
try:
    import xgboost, sklearn
except ImportError:
    !pip install -q scikit-learn xgboost pandas numpy matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# Set plotting aesthetic
plt.rcParams.update({"font.family": "sans-serif", "figure.dpi": 120})
print("All scientific packages loaded successfully!")

## 2. Dataset Ingestion & Physical Exploration

In [ ]:
# Load dataset from repository or generate authentic distribution
import os
csv_path = "../data/gas_turbine_ambient_dataset.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    # Direct GitHub raw link fallback for Colab
    url = "https://raw.githubusercontent.com/mohammedbasra-star/gas-turbine-ambient-effects-regression/main/data/gas_turbine_ambient_dataset.csv"
    try:
        df = pd.read_csv(url)
    except Exception:
        from data.dataset import load_gas_turbine_data
        df = load_gas_turbine_data()

print(f"Total Operational Observations: {len(df):,}")
df.head()

In [ ]:
# Correlation Analysis with Target Power Output (PE)
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Thermodynamic Correlation Matrix", fontweight="bold")
plt.show()

Notice the strong negative correlation ($r \approx -0.95$) between Ambient Temperature (**AT**) and Net Power (**PE**), confirming compressor inlet air density as the governing physical mechanism.

## 3. Model Training & Benchmarking

In [ ]:
X = df[["AT", "V", "AP", "RH"]].values
y = df["PE"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

candidate_models = {
    "Linear Regression (OLS)": LinearRegression(),
    "Polynomial Ridge (Degree 2)": Pipeline([("poly", PolynomialFeatures(degree=2)), ("ridge", Ridge(alpha=10.0))]),
    "Random Forest (100 Trees)": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=150, learning_rate=0.06, max_depth=5, random_state=42, n_jobs=-1)
}

results = []
for name, model in candidate_models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    rmse = np.sqrt(np.mean((y_test - preds) ** 2))
    mae = np.mean(np.abs(y_test - preds))
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100.0
    ss_res = np.sum((y_test - preds) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2 = 1.0 - (ss_res / ss_tot)
    
    results.append({"Model": name, "R2": r2, "RMSE (MW)": rmse, "MAE (MW)": mae, "MAPE (%)": mape})

results_df = pd.DataFrame(results)
results_df.style.highlight_max(subset=["R2"], color="#C8E6C9").highlight_min(subset=["RMSE (MW)", "MAE (MW)"], color="#C8E6C9")

## 4. Visualizing Ambient Derating & Parity

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["AT"], df["PE"], c=df["V"], cmap="plasma", alpha=0.35, s=15, label="Plant Observations")
cbar = plt.colorbar()
cbar.set_label("Exhaust Vacuum V (cm Hg)")

# Overlay fitted regression trend
at_range = np.linspace(df["AT"].min(), df["AT"].max(), 100)
v_mean = np.full_like(at_range, df["V"].mean())
ap_mean = np.full_like(at_range, df["AP"].mean())
rh_mean = np.full_like(at_range, df["RH"].mean())
X_curve = scaler.transform(np.column_stack([at_range, v_mean, ap_mean, rh_mean]))
y_curve = candidate_models["XGBoost Regressor"].predict(X_curve)

plt.plot(at_range, y_curve, color="#00E676", linewidth=3, label="Mean Operating Trend (XGBoost)")
plt.axvline(15.0, color="red", linestyle="--", label="ISO Rating Point (15°C)")
plt.xlabel("Ambient Temperature AT (°C)", fontweight="bold")
plt.ylabel("Net Electrical Power PE (MW)", fontweight="bold")
plt.title("Gas Turbine Derating Response Across Ambient Temperatures", fontweight="bold")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 5. Operational Scenario Simulator (Interactive Inference)
Simulate power output under contrasting climatic conditions:

In [ ]:
scenarios = {
    "Winter Cold Peak":   {"AT": 5.0,  "V": 38.0, "AP": 1022.0, "RH": 80.0},
    "ISO Standard":       {"AT": 15.0, "V": 45.0, "AP": 1013.25,"RH": 60.0},
    "Summer Heatwave":    {"AT": 35.0, "V": 72.0, "AP": 1005.0, "RH": 35.0},
}

best_model = candidate_models["XGBoost Regressor"]
print(f"{'Scenario':<20} | {'AT (°C)':<8} | {'Predicted Power (MW)':<22} | {'Derating vs ISO'}")
print("-" * 70)

iso_power = best_model.predict(scaler.transform([[15.0, 45.0, 1013.25, 60.0]]))[0]

for s_name, cond in scenarios.items():
    cond_scaled = scaler.transform([[cond["AT"], cond["V"], cond["AP"], cond["RH"]]])
    p_pred = best_model.predict(cond_scaled)[0]
    delta = p_pred - iso_power
    sign = "+" if delta >= 0 else ""
    print(f"{s_name:<20} | {cond['AT']:<8.1f} | {p_pred:<22.2f} | {sign}{delta:.2f} MW ({sign}{(delta/iso_power)*100:.1f}%)")